In [0]:

df = spark.read.format("xml") \
    .option("rowTag", "xs:simpleType") \
    .option("valueTag", "simpleType_value") \
    .load('/mnt/dplandingstoragetest/brd_example/xsd/iec62325-451-6-generationload_v3_0.xsd')

In [0]:

from pyspark.sql.functions import input_file_name, explode, inline, explode_outer

df = spark.read.format("xml") \
    .option("rowTag", "xs:schema") \
    .option("valueTag", "value_tag") \
    .load('/mnt/dplandingstoragetest/brd_example/xsd/iec62325-451-6-generationload_v3_0.xsd')

cols_to_drop = [c for c in df.columns if c.startswith("_")]
df = df.drop(*cols_to_drop)

rename_map = {c: c.replace("xs:", "") for c in df.columns}
df = df.withColumnsRenamed(rename_map)
df = df.withColumn("element_type", df["element._type"])
df = df.withColumn("schema_location", df["import._schemaLocation"])
df = df.withColumn("file_name", input_file_name())
df = df.drop(*["import", "element"])

# complex part
df_complex = df.withColumn("complex_type", explode_outer("complexType"))
df_complex = df_complex.withColumn("name", df_complex["complex_type._name"])
df_complex = df_complex.withColumn("complex_type_sequence", explode_outer("complex_type.xs:sequence.xs:element")) # sequence and simpleContent are mutually exclusive
df_complex = df_complex.withColumn("complex_type_sequence_max_occurs", df_complex["complex_type_sequence._maxOccurs"])
df_complex = df_complex.withColumn("complex_type_sequence_min_occurs", df_complex["complex_type_sequence._minOccurs"])
df_complex = df_complex.withColumn("complex_type_sequence_name", df_complex["complex_type_sequence._name"])
df_complex = df_complex.withColumn("complex_type_sequence_model_reference", df_complex["complex_type_sequence._sawsdl:modelReference"])
df_complex = df_complex.withColumn("complex_type_sequence_type", df_complex["complex_type_sequence._type"])

df_complex = df_complex.withColumn("complex_type_simple_base", df_complex["complex_type.xs:simpleContent.xs:extension._base"])
df_complex = df_complex.withColumn("complex_type_simple_use", df_complex["complex_type.xs:simpleContent.xs:extension.xs:attribute._use"])
df_complex = df_complex.drop(*["complexType", "simpleType", "complex_type", "complex_type_sequence"])

# simple part
df_simple = df.withColumn("simple_type", explode_outer("simpleType"))
df_simple = df_simple.withColumn("simple_type_name", df_simple["simple_type._name"])
df_simple = df_simple.withColumn("simple_type_model_reference", df_simple["simple_type._sawsdl:modelReference"])
df_simple = df_simple.withColumn("simple_type_max_inclusive", df_simple["simple_type.xs:restriction.xs:maxInclusive._value"])
df_simple = df_simple.withColumn("simple_type_max_length", df_simple["simple_type.xs:restriction.xs:maxLength._value"])
df_simple = df_simple.withColumn("simple_type_min_inclusive", df_simple["simple_type.xs:restriction.xs:minInclusive._value"])
df_simple = df_simple.withColumn("simple_type_pattern", df_simple["simple_type.xs:restriction.xs:pattern._value"])
df_simple = df_simple.drop(*["complexType", "simpleType", "simple_type"])

df_complex.createOrReplaceTempView("complex")
df_simple.createOrReplaceTempView("simple")


In [0]:
%sql
SELECT * FROM complex order by name;

In [0]:
%sql
SELECT * FROM simple order by 2

In [0]:
df_xml = spark.read.format("xml").option("rowTag", "GL_MarketDocument").option("primitivesAsString", "true").load("/mnt/dplandingstoragetest/brd_example/data/a16abc-example.xml")
display(df_xml)


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, DateType, TimestampType

xsd_to_spark = {
    "string": StringType(),
    "int": IntegerType(),
    "decimal": DoubleType(),
    "boolean": BooleanType(),
    "date": DateType(),
    "dateTime": TimestampType()
}

def build_struct(xsd_element):
    fields = []
    for child in xsd_element.type.content.iter_elements():
        name = child.name
        xsd_type = child.type.local_name
        
        if child.type.is_complex():
            # Nested structure → recurse
            field_type = build_struct(child)
        else:
            # Primitive type
            field_type = xsd_to_spark.get(xsd_type, StringType())
        
        fields.append(StructField(name, field_type, True))
    
    return StructType(fields)


# COMMAND ----------

root_element = schema.elements['RootElementName']  # replace with your XML root
spark_schema = build_struct(root_element)
print(spark_schema.json())